# Convert Reviews to Sentiment Labels

This notebook converts a numeric rating scale into sentiment labels for text classification.

### Two Modes

**Binary mode** (`binary_mode = True`, default):
- **positive** = high ratings (top group)
- **negative** = low ratings (bottom group)
- Neutral ratings (middle value in odd-numbered scales) are dropped as too ambiguous

**Multiclass mode** (`binary_mode = False`):
- Each rating value becomes its own label (e.g., `rating_1`, `rating_2`, ..., `rating_5`)
- No ratings are dropped

The notebook **auto-detects** the rating column and builds the label mapping automatically. You only need to set your CSV file name and choose binary or multiclass mode.

**Run this notebook first**, before the Data Preparation notebook.

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# CHANGE THESE VALUES FOR YOUR DATASET
# ============================================================

input_file = "coffee_maker.csv"        # Your CSV file name
binary_mode = True                     # True = positive/negative only
                                       # False = keep all ratings as separate labels

# ============================================================
# DO NOT CHANGE ANYTHING BELOW THIS LINE
# ============================================================

df = pd.read_csv(input_file)
original_count = len(df)
print(f"Loaded: {len(df)} rows, {len(df.columns)} columns")
print(f"Columns: {df.columns.tolist()}")
print(f"Mode: {'Binary' if binary_mode else 'Multiclass'}\n")

# ---- Auto-detect the rating column ----
# Look for a numeric column with 2-10 unique integer-like values (a rating scale).
# Prefer columns whose name contains common rating keywords.
rating_keywords = ["rating", "score", "star", "stars", "rank", "grade", "rate"]
candidates = []
for col in df.select_dtypes(include=[np.number]).columns:
    unique_vals = df[col].dropna().unique()
    if 2 <= len(unique_vals) <= 10:
        if all(float(v) == int(float(v)) for v in unique_vals):
            name_match = any(kw in col.lower() for kw in rating_keywords)
            candidates.append((col, len(unique_vals), name_match))

if not candidates:
    print("ERROR: Could not auto-detect a rating column.")
    print("No numeric column with 2-10 unique integer values was found.")
    raise ValueError("Rating column not found. Check your dataset.")

# Prefer keyword matches, then fewest unique values (simplest scale)
candidates.sort(key=lambda x: (not x[2], x[1]))
rating_column = candidates[0][0]
unique_ratings = sorted(df[rating_column].dropna().unique())
print(f"Auto-detected rating column: '{rating_column}'")
print(f"Rating values found: {unique_ratings}")

# ---- Build label mapping based on mode ----
if binary_mode:
    # Split into bottom half (negative) and top half (positive).
    # If odd number of ratings, the middle value is dropped as neutral.
    n = len(unique_ratings)
    mid = n // 2
    negative_ratings = unique_ratings[:mid]
    positive_ratings = unique_ratings[mid + (n % 2):]
    neutral_ratings = [unique_ratings[mid]] if n % 2 == 1 else []

    label_map = {}
    for r in negative_ratings:
        label_map[r] = "negative"
    for r in positive_ratings:
        label_map[r] = "positive"

    print(f"\nSentiment mapping:")
    print(f"  Negative ({len(negative_ratings)}): {negative_ratings}")
    print(f"  Positive ({len(positive_ratings)}): {positive_ratings}")
    if neutral_ratings:
        print(f"  Dropped as neutral: {neutral_ratings}")

    output_file = input_file.replace(".csv", "_binary.csv")

else:
    # Multiclass: each rating becomes its own label (e.g., "rating_1")
    label_map = {r: f"rating_{int(r)}" for r in unique_ratings}

    print(f"\nMulticlass mapping:")
    for r, label in label_map.items():
        count = len(df[df[rating_column] == r])
        print(f"  {r} -> '{label}' ({count:,} rows)")

    output_file = input_file.replace(".csv", "_multiclass.csv")

print()

# ---- Apply the mapping ----
df = df[df[rating_column].isin(label_map.keys())].copy()
df["sentiment"] = df[rating_column].map(label_map)

# Drop the original rating column, keep sentiment as the new label.
df = df.drop(columns=[rating_column])

# Show results
rows_dropped = original_count - len(df)
if rows_dropped > 0:
    print(f"Rows dropped: {rows_dropped}")
print(f"Final dataset: {len(df)} rows\n")
print("Class distribution:")
for label, count in df["sentiment"].value_counts().items():
    pct = count / len(df) * 100
    bar = "*" * int(pct / 2)
    print(f"  {label}: {count:,} ({pct:.1f}%) {bar}")

# Save to new file
df.to_csv(output_file, index=False)
print(f"\nSaved to: {output_file}")